In [ ]:
import requests

url = "https://raw.githubusercontent.com/lucasaraujomedeiros/llm-lab/main/frankenstein.txt"
file_path = "frankenstein.txt"

response = requests.get(url, timeout=30)
response.raise_for_status()

with open(file_path, "wb") as f:
    f.write(response.content)


In [ ]:
with open("frankenstein.txt", encoding="utf-8") as f:
    raw_text = f.read()

# Tokenização

Definição de um tokenizador simples (separa apenas com o .split() de python) e possui token para palavras desconhecidas

In [ ]:
def getVocab(text):
    tokens = sorted(set(tokenizingMethod(text)))
    if "<UNKNOWN>" not in tokens:
      tokens.append("<UNKNOWN>")
    return {string: num for (num,string) in enumerate(tokens)}


def tokenizingMethod(text):
    return text.split()

class Tokenizer:
    def __init__(self, vocab):
        self.vocab = vocab
        self.decodeDict = {string: num for (num, string) in vocab.items()}

    def encode(self, text):
        ids = [self.vocab[word] if word in self.vocab else self.vocab["<UNKNOWN>"] for word in tokenizingMethod(text)]
        return ids

    def decode(self, ids):
        words = [self.decodeDict[num] for num in ids]
        return words

In [ ]:
vocab = getVocab(raw_text)
simpleTokenizer = Tokenizer(vocab)

In [ ]:
text = raw_text[:50]
print("Texto: " + text + "\nFim do texto\n")
ids = simpleTokenizer.encode(text)
print(ids)

Texto: Frankenstein,

or the Modern Prometheus


by

Mary
Fim do texto

[559, 7637, 10369, 779, 892, 2368, 762]


Decodificando os ids para texto de volta:

In [ ]:
print(simpleTokenizer.decode(ids))

['Frankenstein,', 'or', 'the', 'Modern', 'Prometheus', 'by', 'Mary']


Aplicando o encoding para um texto contendo palavras fora do vocabulário:

In [ ]:
text = "It is spring, moonless night in the small town, starless and bible-black, the cobblestreets silent and the hunched, courters’ -and -rabbits’ wood limping invisible down to the sloeblack, slow, black, crowblack, fishingboat-bobbing sea. The houses are blind as moles (though moles see fine to-night in the snouting, velvet dingles) or blind as Captain Cat there in the muffled middle by the pump and the town clock, the shops in mourning, the Welfare Hall in widows’ weeds. And all the people of the lulled and dumbfounded town are sleeping now."
print("Texto: " + text + "\nFim do texto\n")
ids = simpleTokenizer.encode(text)
print(ids)

Texto: It is spring, moonless night in the small town, starless and bible-black, the cobblestreets silent and the hunched, courters’ -and -rabbits’ wood limping invisible down to the sloeblack, slow, black, crowblack, fishingboat-bobbing sea. The houses are blind as moles (though moles see fine to-night in the snouting, velvet dingles) or blind as Captain Cat there in the muffled middle by the pump and the town clock, the shops in mourning, the Welfare Hall in widows’ weeds. And all the people of the lulled and dumbfounded town are sleeping now.
Fim do texto

[666, 6306, 9856, 11604, 7395, 5949, 10369, 9640, 10595, 11604, 1524, 11604, 10369, 11604, 9537, 1524, 10369, 11604, 11604, 11604, 11604, 11441, 11604, 6283, 3917, 10519, 10369, 11604, 11604, 2145, 11604, 11604, 9261, 1011, 5740, 1666, 2168, 1725, 11604, 11604, 11604, 9301, 4787, 11604, 5949, 10369, 11604, 11604, 11604, 7637, 2168, 1725, 404, 11604, 10398, 5949, 10369, 11604, 7007, 2368, 10369, 11604, 1524, 10369, 10594, 2675, 103

In [ ]:
print(simpleTokenizer.decode(ids))

['It', 'is', 'spring,', '[UNKNOWN]', 'night', 'in', 'the', 'small', 'town,', '[UNKNOWN]', 'and', '[UNKNOWN]', 'the', '[UNKNOWN]', 'silent', 'and', 'the', '[UNKNOWN]', '[UNKNOWN]', '[UNKNOWN]', '[UNKNOWN]', 'wood', '[UNKNOWN]', 'invisible', 'down', 'to', 'the', '[UNKNOWN]', '[UNKNOWN]', 'black,', '[UNKNOWN]', '[UNKNOWN]', 'sea.', 'The', 'houses', 'are', 'blind', 'as', '[UNKNOWN]', '[UNKNOWN]', '[UNKNOWN]', 'see', 'fine', '[UNKNOWN]', 'in', 'the', '[UNKNOWN]', '[UNKNOWN]', '[UNKNOWN]', 'or', 'blind', 'as', 'Captain', '[UNKNOWN]', 'there', 'in', 'the', '[UNKNOWN]', 'middle', 'by', 'the', '[UNKNOWN]', 'and', 'the', 'town', 'clock,', 'the', '[UNKNOWN]', 'in', 'mourning,', 'the', '[UNKNOWN]', '[UNKNOWN]', 'in', '[UNKNOWN]', '[UNKNOWN]', 'And', 'all', 'the', 'people', 'of', 'the', 'lulled', 'and', '[UNKNOWN]', 'town', 'are', 'sleeping', '[UNKNOWN]']


# Tokenização bytepair encoding


Para evitar a perda de sentido quando codificamos várias palavras diferentes para um mesmo id representando "UNKNOWN", podemos usar a estratégia de quebrar palavras desconhecidas em pequenos tokens já mapeados para o dicionário. Esse método recebe o nome de bytepair encoding.

In [ ]:
import tiktoken
import torch

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
enc = tokenizer.encode("Lucas Araujo joga bola.")
print(enc)

for token in enc:
  print(tokenizer.decode([token]))

[22946, 292, 30574, 84, 7639, 474, 10949, 275, 5708, 13]
Luc
as
 Ara
u
jo
 j
oga
 b
ola
.


In [ ]:
enc = tokenizer.encode("The cat is watching the Fluminense game.")
print(enc)

for token in enc:
  print(tokenizer.decode([token]))

[464, 3797, 318, 4964, 262, 1610, 7230, 1072, 983, 13]
The
 cat
 is
 watching
 the
 Fl
umin
ense
 game
.


Note que usamos a biblioteca Tiktoken e, no caso das palavras em português, elas foram quebradas em vários tokens menores, justamente por não constarem no vocabulário do tokenizador. Isso pode ser um problema pois a palavra "bola", que por si só tem um significado específico, será codificada em dois tokens diferentes, "b" e "ola".

# DataLoader e Dataset


In [ ]:
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader

In [ ]:
class MyDataset(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        tokens = tokenizer.encode(txt)

        for i in range(0, len(tokens) - max_length - 2, stride):
            self.input_ids.append(torch.tensor(tokens[i:i+max_length]))
            self.target_ids.append(torch.tensor(tokens[i+1:i+max_length+1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        return self.input_ids[i], self.target_ids[i]

def create_dataloader(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = MyDataset(txt, tokenizer, max_length, stride)

    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

In [ ]:
mydataloader = create_dataloader(raw_text, shuffle=False)
data_iter = iter(mydataloader)

input, target = next(data_iter)
print("input:\n", input, "\ntarget:\n", target)

input:
 tensor([[17439, 37975,    11,  ...,   198, 36996,   286],
        [  314,  2513,   287,  ...,   287,   883,  3318],
        [ 8737,   290, 10974,  ...,   198,    75,  4820],
        [41168,   198,    82,  ...,   198,  4711, 35066]]) 
target:
 tensor([[37975,    11,   198,  ..., 36996,   286,  8737],
        [ 2513,   287,   262,  ...,   883,  3318, 41168],
        [  290, 10974,    13,  ...,    75,  4820,   699],
        [  198,    82,  6212,  ...,  4711, 35066,   423]])
